# 📊 01 — Exploratory Data Analysis (EDA)
> Notebook ini mencakup eksplorasi awal dataset: tipe data, statistik deskriptif,
> missing values, distribusi target, korelasi, dan distribusi fitur kategorikal.

## 📦 Import Library

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('..')  # agar src/ bisa diimpor

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_config, load_raw_data

pd.set_option('display.max_columns', None)
plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid', palette='muted')

print('✅ Library berhasil diimpor.')

## 📂 Memuat Dataset
Dataset memuat 13 fitur utama untuk prediksi harga jual rumah.

| Kolom | Deskripsi |
|---|---|
| `Id` | Identifikasi unik |
| `MSSubClass` | Tipe bangunan |
| `MSZoning` | Zona properti |
| `LotArea` | Luas lahan (sqft) |
| `LotConfig` | Konfigurasi lahan |
| `BldgType` | Tipe bangunan |
| `OverallCond` | Rating kondisi |
| `YearBuilt` | Tahun konstruksi |
| `YearRemodAdd` | Tahun renovasi |
| `Exterior1st` | Material eksterior |
| `BsmtFinSF2` | Luas basement tipe 2 |
| `TotalBsmtSF` | Total luas basement |
| `SalePrice` | ⭐ Target: harga jual |

In [ ]:
cfg = load_config('../config.yaml')
dataset = load_raw_data(cfg['data']['raw_path'])
dataset.head(10)

## 🔍 3.1 Informasi Tipe Data

In [ ]:
object_cols = dataset.select_dtypes(include=['object']).columns.tolist()
int_cols    = dataset.select_dtypes(include=['int64']).columns.tolist()
float_cols  = dataset.select_dtypes(include=['float64']).columns.tolist()

print(f'🔤 Kategorikal : {len(object_cols)} → {object_cols}')
print(f'🔢 Integer     : {len(int_cols)}  → {int_cols}')
print(f'🔢 Float       : {len(float_cols)}  → {float_cols}')
dataset.info()

## 📋 3.2 Statistik Deskriptif

In [ ]:
dataset.describe(include='all').T

## ❓ 3.3 Missing Values

In [ ]:
missing = dataset.isnull().sum()
missing = missing[missing > 0].reset_index()
missing.columns = ['Kolom', 'Jumlah Missing']
missing['Persentase (%)'] = (missing['Jumlah Missing'] / len(dataset) * 100).round(2)
print('📋 Kolom dengan nilai kosong:')
print(missing.to_string(index=False))

if not missing.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(data=missing, x='Kolom', y='Persentase (%)', palette='Reds_r', ax=ax)
    ax.set_title('Persentase Missing Values per Kolom', fontsize=13, fontweight='bold')
    for bar, pct in zip(ax.patches, missing['Persentase (%)']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{pct}%', ha='center', fontsize=10)
    plt.tight_layout()
    plt.savefig('../outputs/figures/missing_values.png', bbox_inches='tight')
    plt.show()

## 📈 3.4 Distribusi Target: SalePrice

In [ ]:
train_data = dataset.dropna(subset=['SalePrice'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(train_data['SalePrice'], bins=50, kde=True, color='steelblue', ax=axes[0])
axes[0].set_title('Distribusi SalePrice', fontsize=13, fontweight='bold')
axes[0].axvline(train_data['SalePrice'].mean(), color='red', linestyle='--',
                label=f"Mean: ${train_data['SalePrice'].mean():,.0f}")
axes[0].legend()

sns.boxplot(y=train_data['SalePrice'], color='lightcoral', ax=axes[1])
axes[1].set_title('Boxplot SalePrice', fontsize=13, fontweight='bold')

plt.suptitle('Analisis Distribusi SalePrice', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/saleprice_distribution.png', bbox_inches='tight')
plt.show()

print(f"Mean   : ${train_data['SalePrice'].mean():>12,.0f}")
print(f"Median : ${train_data['SalePrice'].median():>12,.0f}")
print(f"Std    : ${train_data['SalePrice'].std():>12,.0f}")

## 🔥 3.5 Correlation Matrix (Fitur Numerik)

In [ ]:
numerical = train_data.select_dtypes(include=['int64', 'float64'])
corr = numerical.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(12, 8))
sns.heatmap(corr, mask=mask, annot=True, cmap='coolwarm',
            fmt='.2f', linewidths=0.5, linecolor='white',
            vmin=-1, vmax=1, square=True)
plt.title('Correlation Matrix (Fitur Numerik)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/correlation_matrix.png', bbox_inches='tight')
plt.show()

print('\n📈 Korelasi teratas dengan SalePrice:')
print(corr['SalePrice'].drop('SalePrice').sort_values(ascending=False).to_string())

## 🏷️ 3.6 Distribusi Fitur Kategorikal

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(object_cols):
    counts = dataset[col].value_counts()
    sns.barplot(x=counts.index, y=counts.values, palette='viridis', ax=axes[i])
    axes[i].set_title(f'Distribusi: {col}', fontsize=12, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=45)
    for bar in axes[i].patches:
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     int(bar.get_height()), ha='center', va='bottom', fontsize=8)

plt.suptitle('Distribusi Fitur Kategorikal', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/categorical_distribution.png', bbox_inches='tight')
plt.show()

## 🔵 3.7 Scatter Plot: Fitur Numerik vs SalePrice

In [ ]:
num_features = [c for c in int_cols if c not in ['Id', 'SalePrice']]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(num_features):
    axes[i].scatter(train_data[col], train_data['SalePrice'],
                    alpha=0.4, color='steelblue', edgecolors='none', s=15)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('SalePrice')
    axes[i].set_title(f'{col} vs SalePrice', fontsize=10, fontweight='bold')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Scatter Plot: Fitur Numerik vs SalePrice', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/scatter_plots.png', bbox_inches='tight')
plt.show()